In [0]:
# ============================================================

# GOLD – dim_team

# Grain: (source_user_id, project, team_name)

# ============================================================

from pyspark.sql import functions as F

from pyspark.sql.window import Window

# ---------------- CONFIG ----------------

SILVER_TABLE = "workspace.slainte_silver.easyvista_tickets_silver_demo"

GOLD_DB = "workspace.slainte_gold"

DIM_TEAM = f"{GOLD_DB}.dim_team"

# ---------------- SETUP ----------------

spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_src = spark.table(SILVER_TABLE)

# ============================================================

# 1️⃣ Extract team base

# ============================================================

df_team_base = (

    df_src

    .select(

        F.col("source_user_id"),

        F.col("project"),

        F.col("support_group").alias("team_name")

    )

    .filter(

        F.col("source_user_id").isNotNull() &

        F.col("project").isNotNull() &

        F.col("team_name").isNotNull()

    )

    .distinct()

)

# ============================================================

# 2️⃣ Generate surrogate key

# ============================================================

team_window = Window.partitionBy(

    "source_user_id", "project"

).orderBy("team_name")

df_dim_team = (

    df_team_base

    .withColumn("team_id", F.row_number().over(team_window))

    .withColumn("created_at", F.current_timestamp())

    .select(

        "team_id",

        "source_user_id",

        "project",

        "team_name"

    )

)

# ============================================================

# 3️⃣ Write GOLD dimension

# ============================================================

df_dim_team.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_TEAM)

# ============================================================

# 4️⃣ Preview

# ============================================================

print("✅ dim_team created successfully")

display(

    spark.table(DIM_TEAM)

         .orderBy("source_user_id", "project", "team_id")

)
 